> **Kaggle Notebook** — Run this notebook on Kaggle with a GPU accelerator.
> 
> ⚙️ **Required:** Go to *Notebook Settings → Internet* and enable **Internet access**
> so that `git clone` and Hugging Face Hub downloads work correctly.

# Adaptive Data Selection & Curriculum Learning for Compute-Efficient LLM Fine-Tuning
## Cloud GPU Execution & Experiment Orchestration Notebook

This notebook orchestrates the complete 2×2 experiment matrix and ablations on a Kaggle GPU instance:

| Exp | Selection | Ordering | Description |
|:---|:---|:---|:---|
| **E1** | 100% (Full) | Random | Baseline (all 49.4k training examples) |
| **E2** | Random 50% | Random | Uniform random subsampling |
| **E3** | Adaptive 50% | Random | Importance-scored selection (Diversity + Complexity + Length) |
| **E4** | Adaptive 50% | Curriculum | **Full Method** (Adaptive Selection + Easy→Hard Curriculum) |
| **E5** | Random 50% | Curriculum | Curriculum on random subset (Isolation test) |
| **Ablation A** | Adaptive 50% | Random | Diversity score only |
| **Ablation B** | Adaptive 50% | Random | Complexity score only |

**Hardware Target:** 1× NVIDIA GPU with $\ge$ 6–16 GB VRAM (e.g., T4, A100, RTX 3090/4090, L4).

In [5]:
# ── Clone repository (requires Internet enabled in Notebook Settings) ────────
import subprocess, sys

# Quick connectivity check before cloning
result = subprocess.run(["curl", "-s", "--max-time", "5", "https://github.com"],
                        capture_output=True)
if result.returncode != 0:
    print("ERROR: Cannot reach github.com.")
    print("  → Go to Notebook Settings (right sidebar) → Internet → turn ON, then re-run.")
    sys.exit(1)

!git clone https://github.com/Bezawit-cloud/efficient-llm-finetuning.git
%cd efficient-llm-finetuning

Cloning into 'efficient-llm-finetuning'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 126 (delta 66), reused 81 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 59.23 KiB | 14.81 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning


In [6]:
# ── Kaggle: Load HF token & set memory allocator env vars ──────────────────
import os

# Load HuggingFace token from Kaggle Secrets (add secret named 'HF_TOKEN' in
# Notebook Settings → Secrets before running)
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = _secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print(f"[WARN] Could not load HF_TOKEN from Kaggle Secrets: {e}")
    print("       Set os.environ['HF_TOKEN'] manually if Hub rate-limiting occurs.")

# Expandable CUDA segments — avoids fragmentation OOM on T4
# (also set inside train_baseline.py and gpu_smoke_test.py as a safety net)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print(f"PYTORCH_CUDA_ALLOC_CONF = {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")

# All outputs will persist under /kaggle/working/
WORK_DIR = "/kaggle/working/efficient-llm-finetuning"
print(f"Working directory: {WORK_DIR}")

[WARN] Could not load HF_TOKEN from Kaggle Secrets: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 131400071 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.
       Set os.environ['HF_TOKEN'] manually if Hub rate-limiting occurs.
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True
Working directory: /kaggle/working/efficient-llm-finetuning


### Step 1: Environment & GPU Verification

In [7]:
# Verify GPU hardware
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")

Fri Aug 21 00:02:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2: Install Dependencies

In [8]:
# Install repository requirements
%pip install -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


### Step 3: Run GPU Smoke Test
Verifies CUDA detection, Qwen2.5-0.5B loading, LoRA parameter attachment, 1 step execution, and peak memory logging.

In [9]:
!python src/gpu_smoke_test.py

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
GPU MIGRATION SMOKE TEST — QWEN2.5-0.5B-INSTRUCT
PyTorch Version:     2.10.0+cu128
CUDA Available:      True
Active Device:       cuda (Tesla T4)
Total GPU VRAM:      14.56 GB
CUDA Version:        12.8
BF16 Supported:      True
-----------------------------------------------------------------
Loading Tokenizer:   Qwen/Qwen2.5-0.5B-Instruct ...
Map: 100%|█████████████████████████████| 32/32 [00:00<00:00, 3780.67 examples/s]
Loading Model:       Qwen/Qwen2.5-0.5B-Instruct on cuda ...
Loading weights: 100%|█| 290/290 [00:00<00:00, 1596.04it/s, Materializing param=
Model loaded in:     0.36s
LoRA Attached:       4,399,104 trainable params (0.88% of 498,431,872)

Executing 1 Training Step (4 micro-batches of 8 samples = 32 samples, checkpointing=True) ...
{'loss': '3.947', 'grad_norm': '12.69', 'learning_rate': '0.0002', 'epoch': '1'}
{'train_runtime': '3.411', 'train

### Step 4: Generate Full 52k Importance Scoring Cache
Computes tri-component scores (Diversity + Complexity + Response Length) on GPU (~45–60s) and caches to `data/scored_alpaca.json`.

In [10]:
import json
from pathlib import Path
from src.utils import load_config, set_seed
from src.data_utils import load_alpaca_dataset
from src.scoring import score_dataset

config = load_config("configs/base_config.yaml")
set_seed(config["seed"])

cache_path = Path("/kaggle/working/efficient-llm-finetuning/data/scored_alpaca.json")
if not cache_path.exists():
    print("Loading Alpaca dataset for scoring...")
    train_ds, _ = load_alpaca_dataset(config)
    train_examples = [dict(ex) for ex in train_ds]
    print(f"Computing importance scores for {len(train_examples)} examples on GPU...")
    scored = score_dataset(train_examples, config)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump(scored, f)
    print(f"Scores successfully cached to {cache_path} ({len(scored)} items).")
else:
    print(f"Cached scores already exist at {cache_path}.")

Cached scores already exist at /kaggle/working/efficient-llm-finetuning/data/scored_alpaca.json.


### Step 5: Execute Primary Experiment Suite (E1 – E5)
Runs all experiments sequentially with fixed seed `42` and identical LoRA configuration.

In [22]:
!python src/train_baseline.py --config configs/exp1_baseline.yaml

2026-08-20 22:19:21 | INFO     | train | === Experiment E1: full_baseline | seed=42 ===
2026-08-20 22:19:21 | INFO     | train | Config: configs/exp1_baseline.yaml
2026-08-20 22:19:21 | INFO     | train | GPU free at experiment start: 12.83 GB / 14.56 GB
2026-08-20 22:19:22 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-20 22:19:23 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-20 22:19:26 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-20 22:19:27 | INFO     | select_and_order | Selection: method=full, fraction=1.0, n_total=49401
2026-08-20 22:19:27 | INFO     | select_and_order | Selected 49401 / 49401 examples (100.0%)
2026-08-20 22:19:27 | INFO     | train | Training on 49401 examples
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-08-20 22:19:33 | INFO     | train | Loading model: Qwen/Qwen2.5-0.5B-Instruct
Loading weights: 100%|█| 290/

In [28]:
!cat /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/outputs/E1_full_baseline/results.json

{
  "experiment_id": "E1",
  "experiment_name": "full_baseline",
  "config_path": "configs/exp1_baseline.yaml",
  "seed": 42,
  "n_train_examples": 49401,
  "n_eval_examples": 2601,
  "data_fraction": 1.0,
  "selection_method": "full",
  "ordering": "random",
  "wall_clock_seconds": 3913.36,
  "wall_clock_minutes": 65.22,
  "peak_gpu_memory_mb": 6887.6,
  "system_ram": {
    "used_mb": 3802.7,
    "total_mb": 32100.1,
    "percent": 13.4
  },
  "trainable_params": 4399104,
  "total_params": 498431872,
  "trainable_pct": 0.8826,
  "eval_loss": 1.1844764947891235,
  "eval_runtime": 63.1181,
  "eval_samples_per_second": 41.208,
  "eval_steps_per_second": 5.165,
  "epoch": 1.0,
  "train_loss": 1.2218394773611752
}

In [11]:
# E2 — Random 50% (random order, 3 epochs)
!python src/train_baseline.py --config configs/exp2_random50.yaml

2026-08-21 00:04:45 | INFO     | train | === Experiment E2: random50_random_order | seed=42 ===
2026-08-21 00:04:45 | INFO     | train | Config: configs/exp2_random50.yaml
2026-08-21 00:04:45 | INFO     | train | GPU free at experiment start: 14.36 GB / 14.56 GB
2026-08-21 00:04:46 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-21 00:04:46 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-21 00:04:50 | INFO     | train | Computing importance scores (this runs once and is cached) ...
2026-08-21 00:04:50 | INFO     | scoring | Scoring 49401 examples (a=0.333, b=0.333, g=0.334)
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-08-21 00:04:58 | INFO     | scoring | Computing embeddings with all-MiniLM-L6-v2 ...
Loading weights: 100%|█| 103/103 [00:00<00:00, 1553.04it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     |

In [12]:
!ls -lh outputs/E2_random50_random/results.json
!cat outputs/E2_random50_random/results.json

-rw-r--r-- 1 root root 726 Aug 21 00:39 outputs/E2_random50_random/results.json
{
  "experiment_id": "E2",
  "experiment_name": "random50_random_order",
  "config_path": "configs/exp2_random50.yaml",
  "seed": 42,
  "n_train_examples": 24700,
  "n_eval_examples": 2601,
  "data_fraction": 0.5,
  "selection_method": "random",
  "ordering": "random",
  "wall_clock_seconds": 1995.79,
  "wall_clock_minutes": 33.26,
  "peak_gpu_memory_mb": 6887.6,
  "system_ram": {
    "used_mb": 3163.6,
    "total_mb": 32100.1,
    "percent": 11.4
  },
  "trainable_params": 4399104,
  "total_params": 498431872,
  "trainable_pct": 0.8826,
  "eval_loss": 1.1957964897155762,
  "eval_runtime": 63.41,
  "eval_samples_per_second": 41.019,
  "eval_steps_per_second": 5.141,
  "epoch": 1.0,
  "train_loss": 1.236928045440832
}

In [25]:
!python src/train_baseline.py --config configs/exp3_adaptive50_random.yaml


2026-08-21 00:47:04 | INFO     | train | === Experiment E3: adaptive50_random_order | seed=42 ===
2026-08-21 00:47:04 | INFO     | train | Config: configs/exp3_adaptive50_random.yaml
2026-08-21 00:47:04 | INFO     | train | GPU free at experiment start: 14.36 GB / 14.56 GB
2026-08-21 00:47:05 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-21 00:47:06 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-21 00:47:09 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-21 00:47:09 | INFO     | select_and_order | Selection: method=adaptive, fraction=0.5, n_total=49401
2026-08-21 00:47:09 | INFO     | select_and_order | Selected 24700 / 49401 examples (50.0%)
2026-08-21 00:47:09 | INFO     | train | Training on 24700 examples
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-08-21 00:47:15 | INFO     | train | Loading model: Qwen/Qwen2.5-0.5B-Instruct
Loading

In [31]:
!cat results/E3_adaptive50_random_results.json

{
  "experiment_id": "E3",
  "experiment_name": "adaptive50_random_order",
  "config_path": "configs/exp3_adaptive50_random.yaml",
  "seed": 42,
  "n_train_examples": 24700,
  "n_eval_examples": 2601,
  "data_fraction": 0.5,
  "selection_method": "adaptive",
  "ordering": "random",
  "wall_clock_seconds": 1703.05,
  "wall_clock_minutes": 28.38,
  "peak_gpu_memory_mb": 6887.5,
  "system_ram": {
    "used_mb": 2818.7,
    "total_mb": 32100.1,
    "percent": 10.3
  },
  "trainable_params": 4399104,
  "total_params": 498431872,
  "trainable_pct": 0.8826,
  "eval_loss": 1.2046293020248413,
  "eval_runtime": 63.4395,
  "eval_samples_per_second": 41.0,
  "eval_steps_per_second": 5.139,
  "epoch": 1.0,
  "train_loss": 1.1542563734894589
}

In [32]:
# E4 — Full Method: Adaptive 50% + Curriculum (easy->hard, 3 epochs)
!python src/train_baseline.py --config configs/exp4_adaptive50_curriculum.yaml


2026-08-21 01:23:09 | INFO     | train | === Experiment E4: adaptive50_curriculum | seed=42 ===
2026-08-21 01:23:09 | INFO     | train | Config: configs/exp4_adaptive50_curriculum.yaml
2026-08-21 01:23:09 | INFO     | train | GPU free at experiment start: 14.36 GB / 14.56 GB
2026-08-21 01:23:10 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-21 01:23:10 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-21 01:23:14 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-21 01:23:14 | INFO     | select_and_order | Selection: method=adaptive, fraction=0.5, n_total=49401
2026-08-21 01:23:14 | INFO     | select_and_order | Selected 24700 / 49401 examples (50.0%)
2026-08-21 01:23:14 | INFO     | select_and_order | Curriculum ordering applied: easy→hard by complexity sub-score
2026-08-21 01:23:14 | INFO     | train | Training on 24700 examples
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.

In [33]:
cat outputs/E4_adaptive50_curriculum/results.json

{
  "experiment_id": "E4",
  "experiment_name": "adaptive50_curriculum",
  "config_path": "configs/exp4_adaptive50_curriculum.yaml",
  "seed": 42,
  "n_train_examples": 24700,
  "n_eval_examples": 2601,
  "data_fraction": 0.5,
  "selection_method": "adaptive",
  "ordering": "curriculum",
  "wall_clock_seconds": 1706.23,
  "wall_clock_minutes": 28.44,
  "peak_gpu_memory_mb": 6887.5,
  "system_ram": {
    "used_mb": 2841.9,
    "total_mb": 32100.1,
    "percent": 10.4
  },
  "trainable_params": 4399104,
  "total_params": 498431872,
  "trainable_pct": 0.8826,
  "eval_loss": 1.2041741609573364,
  "eval_runtime": 63.2834,
  "eval_samples_per_second": 41.101,
  "eval_steps_per_second": 5.151,
  "epoch": 1.0,
  "train_loss": 1.154269584102334
}

In [36]:

# E5 — Random 50% + Curriculum (isolation test, 3 epochs)
!python src/train_baseline.py --config configs/exp5_random50_curriculum.yaml

2026-08-21 01:56:44 | INFO     | train | === Experiment E5: random50_curriculum | seed=42 ===
2026-08-21 01:56:44 | INFO     | train | Config: configs/exp5_random50_curriculum.yaml
2026-08-21 01:56:44 | INFO     | train | GPU free at experiment start: 14.36 GB / 14.56 GB
2026-08-21 01:56:45 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-21 01:56:46 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-21 01:56:49 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-21 01:56:49 | INFO     | select_and_order | Selection: method=random, fraction=0.5, n_total=49401
2026-08-21 01:56:49 | INFO     | select_and_order | Selected 24700 / 49401 examples (50.0%)
2026-08-21 01:56:49 | INFO     | select_and_order | Curriculum ordering applied: easy→hard by complexity sub-score
2026-08-21 01:56:50 | INFO     | train | Training on 24700 examples
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (

In [38]:
ls -lh outputs/*/results.json

-rw-r--r-- 1 root root 726 Aug 21 00:39 outputs/E2_random50_random/results.json
-rw-r--r-- 1 root root 740 Aug 21 01:16 outputs/E3_adaptive50_random/results.json
-rw-r--r-- 1 root root 747 Aug 21 01:53 outputs/E4_adaptive50_curriculum/results.json
-rw-r--r-- 1 root root 739 Aug 21 02:31 outputs/E5_random50_curriculum/results.json


### Step 6: Execute Ablations (A & B)

In [39]:
# Ablation A — Diversity Only (50% subset, 3 epochs)
!python src/train_baseline.py --config configs/ablation_diversity_only.yaml

# Ablation B — Complexity Only (50% subset, 3 epochs)
!python src/train_baseline.py --config configs/ablation_complexity_only.yaml

2026-08-21 02:34:51 | INFO     | train | === Experiment A1: ablation_diversity_only | seed=42 ===
2026-08-21 02:34:51 | INFO     | train | Config: configs/ablation_diversity_only.yaml
2026-08-21 02:34:51 | INFO     | train | GPU free at experiment start: 14.36 GB / 14.56 GB
2026-08-21 02:34:52 | INFO     | data_utils | Loading dataset: tatsu-lab/alpaca
2026-08-21 02:34:53 | INFO     | data_utils | Train: 49401, Eval: 2601
2026-08-21 02:34:56 | INFO     | train | Loading cached scores from data/scored_alpaca.json
2026-08-21 02:34:57 | INFO     | select_and_order | Selection: method=adaptive, fraction=0.5, n_total=49401
2026-08-21 02:34:57 | INFO     | select_and_order | Selected 24700 / 49401 examples (50.0%)
2026-08-21 02:34:57 | INFO     | train | Training on 24700 examples
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
2026-08-21 02:35:03 | INFO     | train | Loading model: Qwen/Qwen2.5-0.5B-Instruct
Loadin

In [40]:
import json, glob, pandas as pd

results = {}

for f in glob.glob("outputs/*/results.json"):
    with open(f) as fp:
        r = json.load(fp)

    exp = r["experiment_id"]

    results[exp] = {
        "eval_loss": r.get("eval_loss"),
        "train_loss": r.get("train_loss"),
        "wall_clock_min": r.get("wall_clock_minutes"),
        "peak_gpu_mb": r.get("peak_gpu_memory_mb"),
        "data_frac": r.get("data_fraction"),
        "selection": r.get("selection_method"),
        "ordering": r.get("ordering"),
    }

df = pd.DataFrame(results).T.sort_index()

display(df)

df.to_csv("experiments/summary_table.csv")

print(df.to_markdown())

,eval_loss,train_loss,wall_clock_min,peak_gpu_mb,data_frac,selection,ordering
A1,1.204624,1.154256,28.37,6887.5,0.5,adaptive,random
A2,1.204628,1.15426,28.31,6887.5,0.5,adaptive,random
E2,1.195796,1.236928,33.26,6887.6,0.5,random,random
E3,1.204629,1.154256,28.38,6887.5,0.5,adaptive,random
E4,1.204174,1.15427,28.44,6887.5,0.5,adaptive,curriculum
E5,1.195706,1.236344,33.24,6887.6,0.5,random,curriculum


OSError: Cannot save file into a non-existent directory: 'experiments'

In [42]:
%cd /kaggle/working/efficient-llm-finetuning

!echo "=== A1 CONFIG ==="
!cat configs/ablation_diversity_only.yaml

!echo ""
!echo "=== A2 CONFIG ==="
!cat configs/ablation_complexity_only.yaml

/kaggle/working/efficient-llm-finetuning
=== A1 CONFIG ===
# Ablation A1 — diversity-only scoring (β=γ=0, α=1.0)
experiment:
  id: "A1"
  name: "ablation_diversity_only"
  description: "Top-50% selected by diversity score only. Ablates the contribution of the diversity component."

scoring:
  alpha: 1.0
  beta:  0.0
  gamma: 0.0

data:
  selection:
    method: "adaptive"
    fraction: 0.50
  ordering: "random"

training:
  output_dir: "outputs/A1_diversity_only"

seed: 42

=== A2 CONFIG ===
# Ablation A2 — complexity-only scoring (α=γ=0, β=1.0)
experiment:
  id: "A2"
  name: "ablation_complexity_only"
  description: "Top-50% selected by complexity score only. Ablates the contribution of the complexity component."

scoring:
  alpha: 0.0
  beta:  1.0
  gamma: 0.0

data:
  selection:
    method: "adaptive"
    fraction: 0.50
  ordering: "random"

training:
  output_dir: "outputs/A2_complexity_only"

seed: 42


In [52]:
import json
import pandas as pd
import glob
import os

ROOT = "/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning"

results = {}

for f in glob.glob(os.path.join(ROOT, "outputs", "*", "results.json")):
    with open(f) as fp:
        r = json.load(fp)

    exp = r["experiment_id"]

    results[exp] = {
        "eval_loss": r.get("eval_loss"),
        "train_loss": r.get("train_loss"),
        "wall_clock_min": r.get("wall_clock_minutes"),
        "peak_gpu_mb": r.get("peak_gpu_memory_mb"),
        "data_frac": r.get("data_fraction"),
        "selection": r.get("selection_method"),
        "ordering": r.get("ordering"),
    }

df = pd.DataFrame(results).T.sort_index()

display(df)

print("\n=== MARKDOWN TABLE ===")
print(df.to_markdown())

# Save authoritative summary
os.makedirs(os.path.join(ROOT, "experiments"), exist_ok=True)
df.to_csv(os.path.join(ROOT, "experiments", "summary_table.csv"))

print("\nSaved:")
print(os.path.join(ROOT, "experiments", "summary_table.csv"))

,eval_loss,train_loss,wall_clock_min,peak_gpu_mb,data_frac,selection,ordering
A1,1.204624,1.154256,28.37,6887.5,0.5,adaptive,random
A2,1.204628,1.15426,28.31,6887.5,0.5,adaptive,random
E2,1.195796,1.236928,33.26,6887.6,0.5,random,random
E3,1.204629,1.154256,28.38,6887.5,0.5,adaptive,random
E4,1.204174,1.15427,28.44,6887.5,0.5,adaptive,curriculum
E5,1.195706,1.236344,33.24,6887.6,0.5,random,curriculum



=== MARKDOWN TABLE ===
|    |   eval_loss |   train_loss |   wall_clock_min |   peak_gpu_mb |   data_frac | selection   | ordering   |
|:---|------------:|-------------:|-----------------:|--------------:|------------:|:------------|:-----------|
| A1 |     1.20462 |      1.15426 |            28.37 |        6887.5 |         0.5 | adaptive    | random     |
| A2 |     1.20463 |      1.15426 |            28.31 |        6887.5 |         0.5 | adaptive    | random     |
| E2 |     1.1958  |      1.23693 |            33.26 |        6887.6 |         0.5 | random      | random     |
| E3 |     1.20463 |      1.15426 |            28.38 |        6887.5 |         0.5 | adaptive    | random     |
| E4 |     1.20417 |      1.15427 |            28.44 |        6887.5 |         0.5 | adaptive    | curriculum |
| E5 |     1.19571 |      1.23634 |            33.24 |        6887.6 |         0.5 | random      | curriculum |

Saved:
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/effici

In [54]:
%cd /kaggle/working/efficient-llm-finetuning

!find /kaggle/working -type f -name "train_*.log" -print

/kaggle/working/efficient-llm-finetuning
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260820_210635.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260820_221919.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260821_004702.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260821_023449.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260821_015642.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260821_030441.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260821_012307.log
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning/efficient-llm-finetuning/logs/train_20260821_000443.log


In [61]:
%cd /kaggle/working/efficient-llm-finetuning

!pwd
!git status --short
!git branch --show-current

/kaggle/working/efficient-llm-finetuning
/kaggle/working/efficient-llm-finetuning
?? efficient-llm-finetuning/
main


In [62]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!pwd
!git status --short
!git branch --show-current
!git remote -v

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
?? efficient-llm-finetuning/
main
origin	https://github.com/Bezawit-cloud/efficient-llm-finetuning.git (fetch)
origin	https://github.com/Bezawit-cloud/efficient-llm-finetuning.git (push)


In [63]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

print("=== CURRENT REPO ===")
!pwd
!ls -la

print("\n=== NESTED DIRECTORY ===")
!ls -la efficient-llm-finetuning

print("\n=== NESTED RESULTS ===")
!find efficient-llm-finetuning -type f -name "results.json" -print | sort

print("\n=== NESTED GIT ===")
!ls -la efficient-llm-finetuning/.git 2>/dev/null || echo "No nested .git"

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
=== CURRENT REPO ===
/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
total 64
drwxr-xr-x 12 root root 4096 Aug 21 00:02 .
drwxr-xr-x  9 root root 4096 Aug 20 21:06 ..
drwxr-xr-x  2 root root 4096 Aug 20 21:03 configs
drwxr-xr-x  2 root root 4096 Aug 20 21:07 data
drwxr-xr-x 12 root root 4096 Aug 21 03:52 efficient-llm-finetuning
drwxr-xr-x  8 root root 4096 Aug 21 04:05 .git
-rw-r--r--  1 root root  675 Aug 20 21:03 .gitignore
drwxr-xr-x  2 root root 4096 Aug 20 22:19 logs
drwxr-xr-x  2 root root 4096 Aug 20 21:03 notebooks
drwxr-xr-x  4 root root 4096 Aug 20 21:07 outputs
drwxr-xr-x  2 root root 4096 Aug 20 21:03 paper
-rw-r--r--  1 root root 7110 Aug 20 21:03 README.md
-rw-r--r--  1 root root  567 Aug 20 21:03 requirements.txt
drwxr-xr-x  2 root root 4096 Aug 20 23:32 results
drwxr-xr-x  3 root root 4096 Aug 20 21:04 src

=== NESTED DIRECTORY ===
total 64
drwxr-xr-x 12 root root 4096 Aug 21 03:52 .
dr

In [64]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!git status --short
!git branch --show-current
!git remote -v

print("\n=== GITIGNORE ===")
!cat .gitignore

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
?? efficient-llm-finetuning/
main
origin	https://github.com/Bezawit-cloud/efficient-llm-finetuning.git (fetch)
origin	https://github.com/Bezawit-cloud/efficient-llm-finetuning.git (push)

=== GITIGNORE ===
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class
*.so

# Environments
.env
.venv
env/
venv/
ENV/
env.bak/
venv.bak/

# Jupyter Notebook checkpoints
.ipynb_checkpoints/
*/.ipynb_checkpoints/*

# Data caches and raw datasets (preserves reproducibility via code)
data/
data/*.json
data/scored_alpaca.json
!data/.gitkeep

# Checkpoint weights and model binaries (keeps repository lightweight)
outputs/
outputs/**/checkpoint-*/
outputs/**/*.safetensors
outputs/**/*.bin
outputs/**/*.pt
outputs/**/*.pth
outputs/**/*.onnx
*.safetensors
*.bin
*.pt

# Execution logs and temporary scratch files
logs/
scratch/
*.log

# OS generated files
.DS_Store
Thumbs.db


In [65]:
print("\n=== CURRENT OUTPUTS ===")
!find outputs -maxdepth 2 -type f -name "results.json" -print | sort

print("\n=== NESTED OUTPUTS ===")
!find efficient-llm-finetuning/outputs -maxdepth 2 -type f -name "results.json" -print | sort


=== CURRENT OUTPUTS ===
outputs/E1_full_baseline/results.json

=== NESTED OUTPUTS ===
efficient-llm-finetuning/outputs/A1_diversity_only/results.json
efficient-llm-finetuning/outputs/A2_complexity_only/results.json
efficient-llm-finetuning/outputs/E2_random50_random/results.json
efficient-llm-finetuning/outputs/E3_adaptive50_random/results.json
efficient-llm-finetuning/outputs/E4_adaptive50_curriculum/results.json
efficient-llm-finetuning/outputs/E5_random50_curriculum/results.json


In [66]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!cp efficient-llm-finetuning/outputs/A1_diversity_only/results.json outputs/A1_diversity_only/results.json
!cp efficient-llm-finetuning/outputs/A2_complexity_only/results.json outputs/A2_complexity_only/results.json
!cp efficient-llm-finetuning/outputs/E2_random50_random/results.json outputs/E2_random50_random/results.json
!cp efficient-llm-finetuning/outputs/E3_adaptive50_random/results.json outputs/E3_adaptive50_random/results.json
!cp efficient-llm-finetuning/outputs/E4_adaptive50_curriculum/results.json outputs/E4_adaptive50_curriculum/results.json
!cp efficient-llm-finetuning/outputs/E5_random50_curriculum/results.json outputs/E5_random50_curriculum/results.json

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
cp: cannot create regular file 'outputs/A1_diversity_only/results.json': No such file or directory
cp: cannot create regular file 'outputs/A2_complexity_only/results.json': No such file or directory
cp: cannot create regular file 'outputs/E2_random50_random/results.json': No such file or directory
cp: cannot create regular file 'outputs/E3_adaptive50_random/results.json': No such file or directory
cp: cannot create regular file 'outputs/E4_adaptive50_curriculum/results.json': No such file or directory
cp: cannot create regular file 'outputs/E5_random50_curriculum/results.json': No such file or directory


In [67]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

# Create the missing experiment output directories
!mkdir -p outputs/A1_diversity_only
!mkdir -p outputs/A2_complexity_only
!mkdir -p outputs/E2_random50_random
!mkdir -p outputs/E3_adaptive50_random
!mkdir -p outputs/E4_adaptive50_curriculum
!mkdir -p outputs/E5_random50_curriculum

# Copy the six result files from the nested repository
!cp efficient-llm-finetuning/outputs/A1_diversity_only/results.json outputs/A1_diversity_only/
!cp efficient-llm-finetuning/outputs/A2_complexity_only/results.json outputs/A2_complexity_only/
!cp efficient-llm-finetuning/outputs/E2_random50_random/results.json outputs/E2_random50_random/
!cp efficient-llm-finetuning/outputs/E3_adaptive50_random/results.json outputs/E3_adaptive50_random/
!cp efficient-llm-finetuning/outputs/E4_adaptive50_curriculum/results.json outputs/E4_adaptive50_curriculum/
!cp efficient-llm-finetuning/outputs/E5_random50_curriculum/results.json outputs/E5_random50_curriculum/

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning


In [68]:
!find outputs -maxdepth 2 -type f -name "results.json" -print | sort

outputs/A1_diversity_only/results.json
outputs/A2_complexity_only/results.json
outputs/E1_full_baseline/results.json
outputs/E2_random50_random/results.json
outputs/E3_adaptive50_random/results.json
outputs/E4_adaptive50_curriculum/results.json
outputs/E5_random50_curriculum/results.json


In [69]:
import json
import glob

print("=== EXPERIMENT VERIFICATION ===")

for f in sorted(glob.glob("outputs/*/results.json")):
    with open(f) as fp:
        r = json.load(fp)

    print(
        f"{r['experiment_id']}: "
        f"eval_loss={r.get('eval_loss')}, "
        f"train_loss={r.get('train_loss')}, "
        f"selection={r.get('selection_method')}, "
        f"ordering={r.get('ordering')}"
    )

=== EXPERIMENT VERIFICATION ===
A1: eval_loss=1.2046235799789429, train_loss=1.1542556817049807, selection=adaptive, ordering=random
A2: eval_loss=1.2046276330947876, train_loss=1.1542601288909122, selection=adaptive, ordering=random
E1: eval_loss=1.1844764947891235, train_loss=1.2218394773611752, selection=full, ordering=random
E2: eval_loss=1.1957964897155762, train_loss=1.236928045440832, selection=random, ordering=random
E3: eval_loss=1.2046293020248413, train_loss=1.1542563734894589, selection=adaptive, ordering=random
E4: eval_loss=1.2041741609573364, train_loss=1.154269584102334, selection=adaptive, ordering=curriculum
E5: eval_loss=1.1957060098648071, train_loss=1.2363436802681247, selection=random, ordering=curriculum


In [71]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

from pathlib import Path

gitignore = Path(".gitignore")
text = gitignore.read_text()

rule = "\n# Keep experiment result metadata under version control\n!outputs/*/results.json\n"

if "!outputs/*/results.json" not in text:
    gitignore.write_text(text.rstrip() + rule)

print(gitignore.read_text())

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class
*.so

# Environments
.env
.venv
env/
venv/
ENV/
env.bak/
venv.bak/

# Jupyter Notebook checkpoints
.ipynb_checkpoints/
*/.ipynb_checkpoints/*

# Data caches and raw datasets (preserves reproducibility via code)
data/
data/*.json
data/scored_alpaca.json
!data/.gitkeep

# Checkpoint weights and model binaries (keeps repository lightweight)
outputs/
outputs/**/checkpoint-*/
outputs/**/*.safetensors
outputs/**/*.bin
outputs/**/*.pt
outputs/**/*.pth
outputs/**/*.onnx
*.safetensors
*.bin
*.pt

# Execution logs and temporary scratch files
logs/
scratch/
*.log

# OS generated files
.DS_Store
Thumbs.db
# Keep experiment result metadata under version control
!outputs/*/results.json



In [72]:
# Keep experiment result metadata under version control
!outputs/*/results.json

/bin/bash: line 1: outputs/A1_diversity_only/results.json: Permission denied


In [73]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning")

# Check all 7 JSON files
experiments = [
    "A1_diversity_only",
    "A2_complexity_only",
    "E1_full_baseline",
    "E2_random50_random",
    "E3_adaptive50_random",
    "E4_adaptive50_curriculum",
    "E5_random50_curriculum",
]

for exp in experiments:
    path = ROOT / "outputs" / exp / "results.json"
    print(exp, "→", path.exists())

    if path.exists():
        with open(path) as f:
            r = json.load(f)
        print("   eval_loss:", r.get("eval_loss"))

A1_diversity_only → True
   eval_loss: 1.2046235799789429
A2_complexity_only → True
   eval_loss: 1.2046276330947876
E1_full_baseline → True
   eval_loss: 1.1844764947891235
E2_random50_random → True
   eval_loss: 1.1957964897155762
E3_adaptive50_random → True
   eval_loss: 1.2046293020248413
E4_adaptive50_curriculum → True
   eval_loss: 1.2041741609573364
E5_random50_curriculum → True
   eval_loss: 1.1957060098648071


In [74]:
from pathlib import Path

gitignore = ROOT / ".gitignore"
text = gitignore.read_text()

# Remove any accidentally inserted broken line if present
lines = text.splitlines()
lines = [
    line for line in lines
    if line.strip() != "!outputs/*/results.json"
]

# Add the correct exception
lines.append("")
lines.append("# Keep experiment result metadata under version control")
lines.append("!outputs/*/results.json")

gitignore.write_text("\n".join(lines) + "\n")

print(gitignore.read_text())

# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class
*.so

# Environments
.env
.venv
env/
venv/
ENV/
env.bak/
venv.bak/

# Jupyter Notebook checkpoints
.ipynb_checkpoints/
*/.ipynb_checkpoints/*

# Data caches and raw datasets (preserves reproducibility via code)
data/
data/*.json
data/scored_alpaca.json
!data/.gitkeep

# Checkpoint weights and model binaries (keeps repository lightweight)
outputs/
outputs/**/checkpoint-*/
outputs/**/*.safetensors
outputs/**/*.bin
outputs/**/*.pt
outputs/**/*.pth
outputs/**/*.onnx
*.safetensors
*.bin
*.pt

# Execution logs and temporary scratch files
logs/
scratch/
*.log

# OS generated files
.DS_Store
Thumbs.db
# Keep experiment result metadata under version control

# Keep experiment result metadata under version control
!outputs/*/results.json



In [75]:
!git status --short --untracked-files=all

 M .gitignore
?? efficient-llm-finetuning/


In [76]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

print("=== Nested repository ===")
!ls -la efficient-llm-finetuning

print("\n=== Nested Git status ===")
!cd efficient-llm-finetuning && git status --short

print("\n=== Outer Git status ===")
!git status --short --untracked-files=all

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
=== Nested repository ===
total 64
drwxr-xr-x 12 root root 4096 Aug 21 03:52 .
drwxr-xr-x 12 root root 4096 Aug 21 00:02 ..
drwxr-xr-x  2 root root 4096 Aug 21 00:02 configs
drwxr-xr-x  2 root root 4096 Aug 21 00:05 data
drwxr-xr-x  2 root root 4096 Aug 21 03:52 experiments
drwxr-xr-x  8 root root 4096 Aug 21 00:44 .git
-rw-r--r--  1 root root  675 Aug 21 00:02 .gitignore
drwxr-xr-x  2 root root 4096 Aug 21 03:04 logs
drwxr-xr-x  2 root root 4096 Aug 21 00:02 notebooks
drwxr-xr-x  9 root root 4096 Aug 21 03:05 outputs
drwxr-xr-x  2 root root 4096 Aug 21 00:02 paper
-rw-r--r--  1 root root 7110 Aug 21 00:02 README.md
-rw-r--r--  1 root root  567 Aug 21 00:02 requirements.txt
drwxr-xr-x  2 root root 4096 Aug 21 01:55 results
drwxr-xr-x  3 root root 4096 Aug 21 00:03 src

=== Nested Git status ===
?? experiments/
?? results/E3_adaptive50_random_results.json
?? results/E4_adaptive50_curriculum_results.json

=== Outer Git sta

In [77]:
print("=== Seven result files ===")
!find outputs -maxdepth 2 -name results.json -print | sort

=== Seven result files ===
outputs/A1_diversity_only/results.json
outputs/A2_complexity_only/results.json
outputs/E1_full_baseline/results.json
outputs/E2_random50_random/results.json
outputs/E3_adaptive50_random/results.json
outputs/E4_adaptive50_curriculum/results.json
outputs/E5_random50_curriculum/results.json


In [78]:
%cd /kaggle/working/efficient-llm-finetuning

!cp -a efficient-llm-finetuning efficient-llm-finetuning_BACKUP

/kaggle/working/efficient-llm-finetuning


In [79]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!rsync -av --exclude='.git/' \
    efficient-llm-finetuning/ \
    ./

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
sending incremental file list
./
.gitignore
README.md
requirements.txt
configs/
configs/README.md
configs/ablation_complexity_only.yaml
configs/ablation_diversity_only.yaml
configs/base_config.yaml
configs/exp1_baseline.yaml
configs/exp2_random50.yaml
configs/exp3_adaptive50_random.yaml
configs/exp4_adaptive50_curriculum.yaml
configs/exp5_random50_curriculum.yaml
data/
data/scored_alpaca.json
experiments/
experiments/summary_table.csv
logs/
logs/data_utils_20260821_000349.log
logs/data_utils_20260821_000445.log
logs/data_utils_20260821_004704.log
logs/data_utils_20260821_012309.log
logs/data_utils_20260821_015644.log
logs/data_utils_20260821_023451.log
logs/data_utils_20260821_030443.log
logs/scoring_20260821_000349.log
logs/scoring_20260821_000450.log
logs/select_and_order_20260821_000516.log
logs/select_and_order_20260821_004709.log
logs/select_and_order_20260821_012314.log
logs/select_and_order_20260821_015649.log
log

In [80]:
!find outputs -maxdepth 2 -type f -name "results.json" -print | sort

outputs/A1_diversity_only/results.json
outputs/A2_complexity_only/results.json
outputs/E1_full_baseline/results.json
outputs/E2_random50_random/results.json
outputs/E3_adaptive50_random/results.json
outputs/E4_adaptive50_curriculum/results.json
outputs/E5_random50_curriculum/results.json


In [81]:
!find experiments -maxdepth 2 -type f -print | sort

experiments/summary_table.csv


In [82]:
!git status --short --untracked-files=all

?? efficient-llm-finetuning/
?? experiments/summary_table.csv
?? results/E2_random50_random_results.json
?? results/E3_adaptive50_random_results.json
?? results/E4_adaptive50_curriculum_results.json


In [83]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

print("=== RESULTS JSONS ===")
!find outputs -maxdepth 2 -type f -name "results.json" -print | sort

print("\n=== EXPERIMENT SUMMARY ===")
!find experiments -maxdepth 2 -type f -print | sort

print("\n=== RESULTS DIRECTORY ===")
!find results -maxdepth 2 -type f -print | sort

print("\n=== SOURCE FILES ===")
!find src -type f -print | sort

print("\n=== CONFIG FILES ===")
!find configs -type f -print | sort

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
=== RESULTS JSONS ===
outputs/A1_diversity_only/results.json
outputs/A2_complexity_only/results.json
outputs/E1_full_baseline/results.json
outputs/E2_random50_random/results.json
outputs/E3_adaptive50_random/results.json
outputs/E4_adaptive50_curriculum/results.json
outputs/E5_random50_curriculum/results.json

=== EXPERIMENT SUMMARY ===
experiments/summary_table.csv

=== RESULTS DIRECTORY ===
results/E1_full_baseline_results.json
results/E2_random50_random_results.json
results/E3_adaptive50_random_results.json
results/E4_adaptive50_curriculum_results.json

=== SOURCE FILES ===
src/data_utils.py
src/gpu_smoke_test.py
src/__pycache__/data_utils.cpython-312.pyc
src/__pycache__/scoring.cpython-312.pyc
src/__pycache__/select_and_order.cpython-312.pyc
src/__pycache__/utils.cpython-312.pyc
src/scoring.py
src/select_and_order.py
src/train_baseline.py
src/utils.py

=== CONFIG FILES ===
configs/ablation_complexity_only.yaml
config

In [84]:
print("\n=== NESTED GIT CHECK ===")
!test -d efficient-llm-finetuning/.git && echo "NESTED .git EXISTS" || echo "NO NESTED .git"


=== NESTED GIT CHECK ===
NESTED .git EXISTS


In [85]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!rm -rf efficient-llm-finetuning/.git

print("Nested Git metadata removed.")

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
Nested Git metadata removed.


In [86]:
!test -d efficient-llm-finetuning/.git && echo "STILL EXISTS" || echo "OK — nested .git removed"

OK — nested .git removed


In [87]:
!git status --short --untracked-files=all

?? efficient-llm-finetuning/.gitignore
?? efficient-llm-finetuning/README.md
?? efficient-llm-finetuning/configs/README.md
?? efficient-llm-finetuning/configs/ablation_complexity_only.yaml
?? efficient-llm-finetuning/configs/ablation_diversity_only.yaml
?? efficient-llm-finetuning/configs/base_config.yaml
?? efficient-llm-finetuning/configs/exp1_baseline.yaml
?? efficient-llm-finetuning/configs/exp2_random50.yaml
?? efficient-llm-finetuning/configs/exp3_adaptive50_random.yaml
?? efficient-llm-finetuning/configs/exp4_adaptive50_curriculum.yaml
?? efficient-llm-finetuning/configs/exp5_random50_curriculum.yaml
?? efficient-llm-finetuning/experiments/summary_table.csv
?? efficient-llm-finetuning/notebooks/cloud_run.ipynb
?? efficient-llm-finetuning/paper/related_work.md
?? efficient-llm-finetuning/requirements.txt
?? efficient-llm-finetuning/results/E2_random50_random_results.json
?? efficient-llm-finetuning/results/E3_adaptive50_random_results.json
?? efficient-llm-finetuning/results/E4_a

In [88]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!rm -rf efficient-llm-finetuning

print("Nested duplicate removed.")

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
Nested duplicate removed.


In [89]:
!find outputs -maxdepth 2 -type f -name "results.json" -print | sort

outputs/A1_diversity_only/results.json
outputs/A2_complexity_only/results.json
outputs/E1_full_baseline/results.json
outputs/E2_random50_random/results.json
outputs/E3_adaptive50_random/results.json
outputs/E4_adaptive50_curriculum/results.json
outputs/E5_random50_curriculum/results.json


In [90]:
!git status --short --untracked-files=all

?? experiments/summary_table.csv
?? results/E2_random50_random_results.json
?? results/E3_adaptive50_random_results.json
?? results/E4_adaptive50_curriculum_results.json


In [91]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!git status --short
!git ls-files | head -100

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning
?? experiments/
?? results/E2_random50_random_results.json
?? results/E3_adaptive50_random_results.json
?? results/E4_adaptive50_curriculum_results.json
.gitignore
README.md
configs/README.md
configs/ablation_complexity_only.yaml
configs/ablation_diversity_only.yaml
configs/base_config.yaml
configs/exp1_baseline.yaml
configs/exp2_random50.yaml
configs/exp3_adaptive50_random.yaml
configs/exp4_adaptive50_curriculum.yaml
configs/exp5_random50_curriculum.yaml
notebooks/cloud_run.ipynb
paper/related_work.md
requirements.txt
results/E1_full_baseline_results.json
src/data_utils.py
src/gpu_smoke_test.py
src/scoring.py
src/select_and_order.py
src/train_baseline.py
src/utils.py


In [92]:
%cd /kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning

!git add .gitignore
!git add experiments/
!git add results/E2_random50_random_results.json
!git add results/E3_adaptive50_random_results.json
!git add results/E4_adaptive50_curriculum_results.json

/kaggle/working/efficient-llm-finetuning/efficient-llm-finetuning


In [94]:
!git add -f \
    outputs/A1_diversity_only/results.json \
    outputs/A2_complexity_only/results.json \
    outputs/E1_full_baseline/results.json \
    outputs/E2_random50_random/results.json \
    outputs/E3_adaptive50_random/results.json \
    outputs/E4_adaptive50_curriculum/results.json \
    outputs/E5_random50_curriculum/results.json

In [95]:
!git status --short

A  experiments/summary_table.csv
A  outputs/A1_diversity_only/results.json
A  outputs/A2_complexity_only/results.json
A  outputs/E1_full_baseline/results.json
A  outputs/E2_random50_random/results.json
A  outputs/E3_adaptive50_random/results.json
A  outputs/E4_adaptive50_curriculum/results.json
A  outputs/E5_random50_curriculum/results.json
A  results/E2_random50_random_results.json
A  results/E3_adaptive50_random_results.json
A  results/E4_adaptive50_curriculum_results.json


In [96]:
!git diff --cached --stat

 experiments/summary_table.csv                 |  7 +++++++
 outputs/A1_diversity_only/results.json        | 28 +++++++++++++++++++++++++++
 outputs/A2_complexity_only/results.json       | 28 +++++++++++++++++++++++++++
 outputs/E1_full_baseline/results.json         | 28 +++++++++++++++++++++++++++
 outputs/E2_random50_random/results.json       | 28 +++++++++++++++++++++++++++
 outputs/E3_adaptive50_random/results.json     | 28 +++++++++++++++++++++++++++
 outputs/E4_adaptive50_curriculum/results.json | 28 +++++++++++++++++++++++++++
 outputs/E5_random50_curriculum/results.json   | 28 +++++++++++++++++++++++++++
 results/E2_random50_random_results.json       | 28 +++++++++++++++++++++++++++
 results/E3_adaptive50_random_results.json     | 28 +++++++++++++++++++++++++++
 results/E4_adaptive50_curriculum_results.json | 28 +++++++++++++++++++++++++++
 11 files changed, 287 insertions(+)


In [97]:
import json
import glob

print("=== STAGED EXPERIMENT RESULTS ===")

for f in sorted(glob.glob("outputs/*/results.json")):
    with open(f) as fp:
        r = json.load(fp)

    print(
        f"{r['experiment_id']}: "
        f"eval_loss={r.get('eval_loss'):.6f}, "
        f"train_loss={r.get('train_loss'):.6f}"
    )

=== STAGED EXPERIMENT RESULTS ===
A1: eval_loss=1.204624, train_loss=1.154256
A2: eval_loss=1.204628, train_loss=1.154260
E1: eval_loss=1.184476, train_loss=1.221839
E2: eval_loss=1.195796, train_loss=1.236928
E3: eval_loss=1.204629, train_loss=1.154256
E4: eval_loss=1.204174, train_loss=1.154270
E5: eval_loss=1.195706, train_loss=1.236344


In [1]:
!git commit -m "results: add all seven experiment results"

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
